# MCMC: Metropolis-Hastings from Scratch

When the posterior has no closed form, we sample from it using **Markov Chain Monte Carlo**.
This notebook covers:
1. **Why MCMC** -- intractable posteriors
2. **Metropolis-Hastings** algorithm step by step
3. **Trace plots** and convergence diagnostics
4. **Effective sample size** and autocorrelation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
np.random.seed(42)
print('Setup complete.')

## 1. The Metropolis-Hastings Algorithm

**Goal:** sample from a target distribution $\pi(\theta)$ that we can evaluate up to a
normalising constant.

**Algorithm:**
1. Start at $\theta_0$
2. Propose $\theta^* \sim q(\theta^* \mid \theta_t)$ (e.g., Gaussian random walk)
3. Acceptance ratio: $\alpha = \min\left(1, \frac{\pi(\theta^*) q(\theta_t \mid \theta^*)}{\pi(\theta_t) q(\theta^* \mid \theta_t)}\right)$
4. Accept $\theta^*$ with probability $\alpha$, else stay at $\theta_t$
5. Repeat

In [ ]:
def metropolis_hastings(log_target, init, n_samples=10000, proposal_std=1.0):
    """
    Metropolis-Hastings with Gaussian random walk proposal.
    log_target: function returning log of unnormalised target density.
    """
    samples = np.zeros(n_samples)
    current = init
    current_log_p = log_target(current)
    accepted = 0

    for i in range(n_samples):
        # Propose
        proposal = current + np.random.normal(0, proposal_std)
        proposal_log_p = log_target(proposal)

        # Acceptance ratio (log scale)
        log_alpha = proposal_log_p - current_log_p

        # Accept or reject
        if np.log(np.random.rand()) < log_alpha:
            current = proposal
            current_log_p = proposal_log_p
            accepted += 1

        samples[i] = current

    acceptance_rate = accepted / n_samples
    return samples, acceptance_rate

print('Metropolis-Hastings sampler defined.')

## 2. Example: Sampling from a Beta Posterior

We use a problem with a known solution (Beta-Binomial) to verify our sampler.

In [ ]:
# Data: 14 heads out of 20 flips
n_flips, n_heads = 20, 14
alpha_prior, beta_prior = 2, 2

# Log unnormalised posterior: log(Beta prior) + log(Binomial likelihood)
def log_posterior(theta):
    if theta <= 0 or theta >= 1:
        return -np.inf
    log_prior = (alpha_prior - 1) * np.log(theta) + (beta_prior - 1) * np.log(1 - theta)
    log_lik = n_heads * np.log(theta) + (n_flips - n_heads) * np.log(1 - theta)
    return log_prior + log_lik

# Run sampler
samples, acc_rate = metropolis_hastings(log_posterior, init=0.5, n_samples=20000, proposal_std=0.1)
burn_in = 2000
samples_post = samples[burn_in:]

print(f'Acceptance rate: {acc_rate:.3f} (target: 0.2-0.5)')
print(f'Posterior mean (MCMC): {samples_post.mean():.4f}')

# Exact posterior
a_post, b_post = alpha_prior + n_heads, beta_prior + n_flips - n_heads
print(f'Posterior mean (exact): {a_post / (a_post + b_post):.4f}')

In [ ]:
# Compare MCMC histogram with exact posterior
theta_grid = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram vs exact
axes[0].hist(samples_post, bins=50, density=True, alpha=0.6, label='MCMC samples')
axes[0].plot(theta_grid, stats.beta.pdf(theta_grid, a_post, b_post), 'r-', lw=2, label='Exact Beta posterior')
axes[0].set_xlabel(r'$\theta$')
axes[0].set_title('MCMC vs Exact Posterior')
axes[0].legend()

# Trace plot
axes[1].plot(samples[:5000], lw=0.3, color='steelblue')
axes[1].axhline(a_post / (a_post + b_post), color='red', ls='--', label='True mean')
axes[1].axvline(burn_in, color='orange', ls='--', label='Burn-in cutoff')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel(r'$\theta$')
axes[1].set_title('Trace Plot')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Convergence Diagnostics

- **Trace plots:** should look like "hairy caterpillars" (good mixing)
- **Autocorrelation:** lower is better; high autocorrelation means inefficient sampling
- **Effective Sample Size (ESS):** number of independent-equivalent samples

In [ ]:
def autocorrelation(x, max_lag=100):
    """Compute autocorrelation up to max_lag."""
    x = x - x.mean()
    n = len(x)
    var = np.var(x)
    acf = np.array([np.sum(x[:n-lag] * x[lag:]) / (n * var) for lag in range(max_lag)])
    return acf

def effective_sample_size(x):
    """Estimate ESS using autocorrelation."""
    acf = autocorrelation(x, max_lag=min(500, len(x) // 2))
    # Sum positive autocorrelations
    n = len(x)
    running_sum = 0
    for k in range(1, len(acf)):
        if acf[k] < 0.05:
            break
        running_sum += acf[k]
    return n / (1 + 2 * running_sum)

# Compare two proposal scales
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, std, label in zip(axes, [0.01, 0.1], ['Too small (σ=0.01)', 'Good (σ=0.1)']):
    s, acc = metropolis_hastings(log_posterior, init=0.5, n_samples=10000, proposal_std=std)
    s = s[2000:]  # burn-in
    acf = autocorrelation(s)
    ess = effective_sample_size(s)
    ax.bar(range(len(acf)), acf, color='steelblue', width=1)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title(f'{label}\nAcc={acc:.2f}, ESS={ess:.0f}')

plt.tight_layout()
plt.show()

## Key Takeaways

- **Metropolis-Hastings** lets us sample from any posterior we can evaluate (up to a constant).
- The **proposal scale** is critical: tune for ~25-45% acceptance.
- Always discard a **burn-in** period and check **trace plots**.
- **ESS** tells you how many independent samples you effectively have.
- More advanced samplers (HMC, NUTS) improve efficiency for high-dimensional problems.

**Next:** Using PyMC for practical Bayesian modelling.